In [ ]:
import pandas as pd
import numpy as np
import re, hashlib
from pathlib import Path

train = pd.read_csv(r"C:\Users\Admin\Documents\2025.1\BA Project\house-prices-advanced-regression-techniques\data\train.csv")   
test = pd.read_csv(r"C:\Users\Admin\Documents\2025.1\BA Project\house-prices-advanced-regression-techniques\data\test.csv")    
raw = pd.read_csv(r"C:\Users\Admin\Documents\2025.1\BA Project\house-prices-advanced-regression-techniques\data\AmesHousing.csv")

def canon(s: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

raw_canon = {canon(c): c for c in raw.columns}

kag_cols = train.columns.tolist()
k2r = {c: raw_canon[canon(c)] for c in kag_cols if canon(c) in raw_canon}

raw2 = raw.rename(columns={v: k for k, v in k2r.items()}).copy()
feature_cols = [c for c in test.columns if c != "Id"]

num_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train[c])]
cat_cols = [c for c in feature_cols if c not in num_cols]

def md5_series(s: pd.Series) -> pd.Series:
    return s.map(lambda x: hashlib.md5(x.encode("utf-8")).hexdigest())

def norm_num_col(s: pd.Series, decimals=3) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce").round(decimals)
    def fmt(v):
        if pd.isna(v):
            return "<NA>"
        if abs(v) < 10**(-decimals):
            v = 0.0
        if float(v).is_integer():
            return str(int(v))
        return f"{v:.{decimals}f}".rstrip("0").rstrip(".")
    return x.map(fmt)

def norm_cat_col(s: pd.Series) -> pd.Series:
    return (s.astype("string")
              .fillna("<NA>")
              .str.strip()
              .str.upper())

def make_sig(df: pd.DataFrame, use_saleprice: bool = False) -> pd.Series:
    parts = []
    for c in num_cols:
        parts.append(norm_num_col(df[c], decimals=3))
    for c in cat_cols:
        parts.append(norm_cat_col(df[c]))

    joined = parts[0].astype(str)
    for p in parts[1:]:
        joined = joined + "|" + p.astype(str)

    if use_saleprice:
        sp = pd.to_numeric(df["SalePrice"], errors="coerce").round(0)
        sp = sp.map(lambda v: "<NA>" if pd.isna(v) else str(int(v)))
        joined = joined + "|" + sp.astype(str)

    return md5_series(joined)

raw2["sig_test"]   = make_sig(raw2, use_saleprice=False)
train_sig_test     = make_sig(train, use_saleprice=False)
test_sig_test      = make_sig(test,  use_saleprice=False)

raw2["sig_train"]  = make_sig(raw2, use_saleprice=True)
train_sig_train    = make_sig(train, use_saleprice=True)

sigtrain_to_pid = raw2.set_index("sig_train")["PID"].to_dict()
train_pid = train_sig_train.map(sigtrain_to_pid)

print("Train missing PID:", train_pid.isna().sum())

cand = raw2.groupby("sig_test")["PID"].apply(list).to_dict()

train_used = pd.DataFrame({"sig_test": train_sig_test, "PID": train_pid})
train_used_map = train_used.dropna().drop_duplicates("sig_test").set_index("sig_test")["PID"].to_dict()

def choose_pid_for_test(sig):
    if sig not in cand:
        return np.nan
    pids = cand[sig]
    if len(pids) == 1:
        return pids[0]
    pid_train = train_used_map.get(sig, None)
    if pid_train is not None:
        remain = [p for p in pids if p != pid_train]
        if len(remain) == 1:
            return remain[0]
        if len(remain) > 1:
            return remain[0] 
    return pids[0]

test_pid = test_sig_test.map(choose_pid_for_test)
print("Test missing PID:", pd.isna(test_pid).sum())

train_with_pid = train.copy()
test_with_pid  = test.copy()

train_with_pid.insert(train_with_pid.columns.get_loc("Id")+1, "PID", train_pid.values)
test_with_pid.insert(test_with_pid.columns.get_loc("Id")+1, "PID", test_pid.values)

train_with_pid.to_csv(r"C:\Users\Admin\Documents\2025.1\BA Project\house-prices-advanced-regression-techniques\data\train_with_PID.csv", index=False)
test_with_pid.to_csv(r"C:\Users\Admin\Documents\2025.1\BA Project\house-prices-advanced-regression-techniques\data\test_with_PID.csv", index=False)
print("Saved to C:\\Users\\Admin\\Documents\\2025.1\\BA Project\\house-prices-advanced-regression-techniques\\data")

Train missing PID: 0
Test missing PID: 19
Saved to C:\Users\Admin\Downloads


In [ ]:
drop_suspects = {
    "MSZoning","Utilities","Exterior1st","Exterior2nd","MasVnrType",
    "KitchenQual","Functional","SaleType","SaleCondition","Electrical",
    "BsmtExposure","BsmtFinType1","BsmtFinType2","GarageType"
}

feature_cols = [c for c in test.columns if c != "Id"]
feature_cols_fallback = [c for c in feature_cols if c not in drop_suspects]

num_cols_fb = [c for c in feature_cols_fallback if pd.api.types.is_numeric_dtype(train[c])]
cat_cols_fb = [c for c in feature_cols_fallback if c not in num_cols_fb]

def norm_num_col2(s: pd.Series, decimals=3) -> pd.Series:
    x = pd.to_numeric(s, errors="coerce").round(decimals)
    return x.map(lambda v: "<NA>" if pd.isna(v) else str(int(v)) if float(v).is_integer() else f"{v:.{decimals}f}".rstrip("0").rstrip("."))

def norm_cat_col2(s: pd.Series) -> pd.Series:
    return (s.astype("string")
              .fillna("<NA>")
              .str.strip()
              .str.upper()
              .replace({"NONE":"<NA>","N/A":"<NA>","NA":"<NA>"}))

def make_sig_cols(df: pd.DataFrame, ncols, ccols) -> pd.Series:
    parts = []
    for c in ncols:
        parts.append(norm_num_col2(df[c], decimals=3))
    for c in ccols:
        parts.append(norm_cat_col2(df[c]))
    joined = parts[0].astype(str)
    for p in parts[1:]:
        joined = joined + "|" + p.astype(str)
    return joined.map(lambda x: hashlib.md5(x.encode("utf-8")).hexdigest())

raw2["sig_fb"]  = make_sig_cols(raw2, num_cols_fb, cat_cols_fb)
test_sig_fb     = make_sig_cols(test,  num_cols_fb, cat_cols_fb)

cand_fb = raw2.groupby("sig_fb")["PID"].apply(list).to_dict()


In [ ]:
test_pid2 = test_pid.copy()

used_pids = set(pd.Series(train_pid).dropna().astype("int64").tolist()) | set(pd.Series(test_pid).dropna().astype("int64").tolist())

missing_idx = test_pid2[pd.isna(test_pid2)].index.tolist()
filled = 0

for i in missing_idx:
    sig = test_sig_fb.iloc[i]
    pids = cand_fb.get(sig, [])
    pids = [p for p in pids if int(p) not in used_pids]
    if len(pids) == 1:
        test_pid2.iloc[i] = pids[0]
        used_pids.add(int(pids[0]))
        filled += 1

print("Filled by fallback signature:", filled)
print("Still missing:", pd.isna(test_pid2).sum())


Filled by fallback signature: 10
Still missing: 9


In [ ]:
block_levels = [
    ["Neighborhood","OverallQual","YearBuilt","GrLivArea","LotArea"],
    ["Neighborhood","OverallQual","YearBuilt","GrLivArea"],
    ["Neighborhood","OverallQual","YearBuilt","LotArea"],
    ["Neighborhood","OverallQual","GrLivArea"],
    ["Neighborhood","YearBuilt","GrLivArea"],
    ["Neighborhood","GrLivArea","LotArea"],
    ["Neighborhood","YearBuilt","OverallQual"],
    ["Neighborhood","GrLivArea"],
    ["Neighborhood"]
]

dist_num = [c for c in [
    "LotArea","OverallQual","OverallCond","YearBuilt","YearRemodAdd",
    "GrLivArea","TotalBsmtSF","1stFlrSF","2ndFlrSF",
    "GarageCars","GarageArea","TotRmsAbvGrd","FullBath","HalfBath",
    "BedroomAbvGr","Fireplaces","MasVnrArea","WoodDeckSF","OpenPorchSF",
    "MoSold","YrSold"
] if c in raw2.columns and c in test.columns]

dist_cat = [c for c in [
    "Neighborhood","MSZoning","BldgType","HouseStyle","Exterior1st","Exterior2nd",
    "Foundation","HeatingQC","KitchenQual","SaleType","SaleCondition"
] if c in raw2.columns and c in test.columns]

BIG = 1e6

def norm_row_num(df, cols):
    out = df[cols].copy()
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

raw_num = norm_row_num(raw2, dist_num)
test_num = norm_row_num(test, dist_num)

mu = raw_num.mean(axis=0)
sd = raw_num.std(axis=0).replace(0, 1.0)

raw_z = (raw_num - mu) / sd
test_z = (test_num - mu) / sd

def cat_equal(a, b):
    a = norm_cat_col2(pd.Series([a])).iloc[0]
    b = norm_cat_col2(pd.Series([b])).iloc[0]
    return a == b

still_missing = test_pid2[pd.isna(test_pid2)].index.tolist()
filled2 = 0

for i in still_missing:
    row = test.iloc[i]

    cand_idx = None
    for cols in block_levels:
        ok = pd.Series([True]*len(raw2))
        for c in cols:
            if c in dist_num:
                v = pd.to_numeric(row[c], errors="coerce")
                rv = pd.to_numeric(raw2[c], errors="coerce")
                ok &= (rv == v) | (rv.isna() & pd.isna(v))
            else:
                v = row[c]
                rv = raw2[c]
                ok &= (norm_cat_col2(rv) == norm_cat_col2(pd.Series([v])).iloc[0])
        idxs = raw2.index[ok].tolist()

        idxs = [j for j in idxs if int(raw2.loc[j, "PID"]) not in used_pids]
        if len(idxs) > 0:
            cand_idx = idxs
            break

    if not cand_idx:
        continue

    kz = test_z.iloc[i].values  
    Rz = raw_z.loc[cand_idx].values 
    num_cost = np.abs(Rz - kz).sum(axis=1)

    cat_pen = np.zeros(len(cand_idx))
    for c in dist_cat:
        v = row[c]
        rv = raw2.loc[cand_idx, c].values
        ve = norm_cat_col2(pd.Series([v])).iloc[0]
        re = norm_cat_col2(pd.Series(rv))
        cat_pen += (re.values != ve).astype(float) * BIG

    cost = num_cost + cat_pen
    best_j = cand_idx[int(np.argmin(cost))]
    best_pid = int(raw2.loc[best_j, "PID"])
    test_pid2.iloc[i] = best_pid
    used_pids.add(best_pid)
    filled2 += 1

print("Filled by block+distance:", filled2)
print("Final missing:", pd.isna(test_pid2).sum())


Filled by block+distance: 9
Final missing: 0


In [ ]:
from datetime import datetime

test_with_pid = test.copy()
test_with_pid.insert(test_with_pid.columns.get_loc("Id")+1, "PID", test_pid2.values)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
out_test = fr"C:\Users\Admin\Downloads\test_with_PID_{ts}.csv"
test_with_pid.to_csv(out_test, index=False, encoding="utf-8-sig")
print("Saved:", out_test)


Saved: C:\Users\Admin\Downloads\test_with_PID_20251212_173742.csv
